# CrashGuard — REASON-stage crash classifier (Lane A training)

Fine-tunes **torchvision `r3d_18`** (Kinetics-400 pretrained 3D CNN) into a binary
**crash / no-crash classifier over 16-frame windows**, then exports TorchScript
weights for the Modal endpoint (`modal_app.py`).

- Runtime: **free Colab T4** (`Runtime → Change runtime type → T4 GPU`), ~30–60 min.
- Output: `crash_r3d18.ts` + `preprocess_contract.json` (download at the end).
- The preprocessing here **must stay identical** to `modal_app.py`: 16 frames,
  112×112, RGB, Kinetics mean/std, tensor shape `(1, C, T, H, W)`.

Deploy after training:
```
modal volume put crashguard-weights crash_r3d18.ts /crash_r3d18.ts
modal deploy modal_app.py
```

In [ ]:
!pip -q install opencv-python-headless scikit-learn
import torch, torchvision
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
assert DEVICE == 'cuda', 'Enable the T4: Runtime -> Change runtime type -> GPU'

## 1. Dataset

The acquisition cell below fills this **folder contract** (everything downstream reads only this):
```
dataset/
  crash/    *.mp4            # clips containing a collision
  normal/   *.mp4            # ordinary driving, no collision
  event_times.json           # {filename: seconds} — when the impact happens (when known)
```

**Primary source — [nexar-ai/nexar_collision_prediction](https://huggingface.co/datasets/nexar-ai/nexar_collision_prediction)** (HuggingFace):
1,500 videos (750 collision / 750 normal), 720p@30fps, with labels and `time_of_event`
metadata. Downloaded per-file over HF's CDN — resumable, cached, **no Google Drive
quota problems**. We subsample `N_PER_CLASS` per class rather than pulling all 31 GB.

**Fallback — CCD** ([Cogito2012/CarCrashDataset](https://github.com/Cogito2012/CarCrashDataset))
via `gdown`; only attempted if the HF path yields nothing (Drive quotas make it flaky).

Honest caveats, so nobody is surprised later:
- Nexar is **dashcam** footage and positives include **near-misses**, so labels are
  slightly noisy for a pure "crash happened" classifier — acceptable for the hackathon
  model; swap in fixed-camera sets (TAD/CADP) via the same folder contract for production.
- **Colab wipes local files on runtime reset.** If that happens, just re-run the
  acquisition cell — it skips anything already present and re-downloads the rest
  (HF cache makes this fast).

In [ ]:
# =============================================================================
# Cell 3 — DATASET ACQUISITION (idempotent; safe to re-run after runtime resets)
#
# Primary  : nexar-ai/nexar_collision_prediction on HuggingFace. Subsampled —
#            never pulls all 31 GB. Hardening learned from real runs:
#              * xet backend disabled (its resume of interrupted files hangs)
#              * every file gets a hard 75s timeout and is skipped on failure
#              * *.incomplete cache leftovers purged first
#              * event times ingested from metadata CSVs even though the label
#                column lives in a DIFFERENT csv (solution.csv)
#              * mass-failure abort -> fallback (HF rate-limits anonymous IPs
#                after a few hundred files; a token below avoids that)
# Fallback : CCD Google Drive folder via gdown (daily quota; capped copies).
#
# Output contract: dataset/crash/*.mp4, dataset/normal/*.mp4, dataset/event_times.json
# =============================================================================
N_PER_CLASS = 85         # videos per class (~3 GB, ~15 min)
# os.environ['HF_TOKEN'] = 'hf_...'   # optional: raises HF rate limits a lot

!pip -q install -U huggingface_hub pandas
import json, os, random, re, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ['HF_HUB_DISABLE_XET'] = '1'   # xet hangs resuming interrupted files
random.seed(7)
VID_EXT = ('.mp4', '.avi', '.mov', '.mkv', '.webm')
Path('dataset/crash').mkdir(parents=True, exist_ok=True)
Path('dataset/normal').mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage('.').free / 1e9
print(f'disk free: {free_gb:.0f} GB')
assert free_gb > 10, 'not enough disk for the subsample — lower N_PER_CLASS'


def have_enough():
    c = len(list(Path('dataset/crash').glob('*.mp4')))
    n = len(list(Path('dataset/normal').glob('*.mp4')))
    print(f'dataset/ currently has {c} crash + {n} normal videos')
    return c >= min(20, N_PER_CLASS) and n >= min(20, N_PER_CLASS)


def acquire_nexar():
    from huggingface_hub import hf_hub_download, list_repo_files
    import pandas as pd

    for p in Path.home().joinpath('.cache/huggingface').rglob('*.incomplete'):
        p.unlink(missing_ok=True)

    REPO = 'nexar-ai/nexar_collision_prediction'
    files = list_repo_files(REPO, repo_type='dataset')
    vids = [f for f in files if f.lower().endswith(VID_EXT)]
    csvs = [f for f in files if f.lower().endswith('.csv')]
    print(f'repo: {len(files)} files | {len(vids)} videos | csvs: {csvs}')

    # labels + event times ingested INDEPENDENTLY — labels live in
    # solution.csv (id,target) while event times live in train/*/metadata.csv
    # (file_name,time_of_event) which has NO label column.
    label_by_id, event_by_id = {}, {}
    for c in csvs:
        try:
            df = pd.read_csv(hf_hub_download(REPO, c, repo_type='dataset'))
        except Exception as e:
            print(f'  {c}: unreadable ({e})'); continue
        print(f'  {c}: columns={list(df.columns)} rows={len(df)}')
        idc  = next((k for k in df.columns if k.lower() in
                     ('id', 'video', 'video_id', 'file', 'filename',
                      'file_name', 'name')), None)
        labc = next((k for k in df.columns if k.lower() in
                     ('label', 'target', 'collision', 'class', 'y')), None)
        tc   = next((k for k in df.columns if 'time' in k.lower()
                     and 'event' in k.lower()), None)
        if not idc:
            continue
        for _, r in df.iterrows():
            key = re.sub(r'\.\w+$', '', str(r[idc])).lstrip('0') or '0'
            if labc is not None:
                label_by_id[key] = int(float(r[labc]) > 0.5)
            if tc is not None and pd.notna(r[tc]):
                event_by_id[key] = float(r[tc])
    print(f'labels from CSV: {len(label_by_id)} | event times: {len(event_by_id)}')

    def label_of(path):
        key = re.sub(r'\.\w+$', '', Path(path).stem).lstrip('0') or '0'
        if key in label_by_id:
            return label_by_id[key], key
        low = path.lower()          # fallback: folder-name convention
        if any(s in low for s in ('positive', 'collision', 'crash', 'accident')):
            return 1, key
        if any(s in low for s in ('negative', 'normal', 'no_collision')):
            return 0, key
        return None, key            # unknown -> never guess a training label

    def grab(rp, budget=75):
        """hf_hub_download with a HARD timeout — a hung transfer gets skipped
        and can never wedge the acquisition (learned the hard way)."""
        return ThreadPoolExecutor(max_workers=1).submit(
            hf_hub_download, REPO, rp, repo_type='dataset').result(timeout=budget)

    train_vids = [v for v in vids if 'test' not in v.lower()]
    pos = [v for v in train_vids if label_of(v)[0] == 1]
    neg = [v for v in train_vids if label_of(v)[0] == 0]
    print(f'labelable videos: {len(pos)} positive, {len(neg)} negative')
    if not pos or not neg:
        return False

    random.shuffle(pos); random.shuffle(neg)
    event_times, done, fail = {}, 0, 0
    for rp, kind in ([(p, 'crash') for p in pos[:N_PER_CLASS]] +
                     [(n, 'normal') for n in neg[:N_PER_CLASS]]):
        dst = Path(f'dataset/{kind}/{Path(rp).name}')
        try:
            if not dst.exists():
                shutil.copy(grab(rp), dst)
            _, key = label_of(rp)
            if kind == 'crash' and key in event_by_id:
                event_times[dst.name] = event_by_id[key]
            done += 1
            if done % 25 == 0:
                print(f'  {done} videos ready...')
        except Exception as e:
            fail += 1
            print(f'  skip {rp}: {type(e).__name__}')
            if fail >= 15 and done < 5:     # mass failures = rate limit
                print('  aborting nexar (likely HF rate limit) -> fallback')
                break
    if event_times:
        json.dump(event_times, open('dataset/event_times.json', 'w'))
    print(f'nexar: {done} ok, {fail} failed | event times: {len(event_times)}')
    return have_enough()


def acquire_ccd(cap_crash=90, cap_normal=120):
    import zipfile
    print('falling back to CCD via gdown (Drive daily quota may block this)...')
    subprocess.run('pip -q install gdown && gdown --folder '
                   'https://drive.google.com/drive/folders/'
                   '1NUwC-bkka0-iPqhEhIgsXWtj0DA2MR-F -O ccd_raw --remaining-ok',
                   shell=True)
    for z in Path('ccd_raw').rglob('*.zip'):
        try: zipfile.ZipFile(z).extractall(z.parent)
        except Exception as e: print('  zip failed:', z.name, e)
    vids = [p for p in Path('ccd_raw').rglob('*') if p.suffix.lower() in VID_EXT]
    print('ccd videos found:', len(vids))
    cr = [v for v in vids if any(s in str(v).lower() for s in
                                 ('crash', 'positive', 'accident'))]
    nm = [v for v in vids if v not in cr]
    random.shuffle(cr); random.shuffle(nm)
    for v in cr[:cap_crash]:                 # capped: don't flood the dataset
        shutil.copy(v, f'dataset/crash/{v.stem}.mp4')
    for v in nm[:cap_normal]:
        shutil.copy(v, f'dataset/normal/{v.stem}.mp4')
    return have_enough()


if have_enough():
    print('dataset already populated — skipping download')
else:
    ok = False
    try:
        ok = acquire_nexar()
    except Exception as e:
        print('nexar path failed:', e)
    if not ok:
        ok = acquire_ccd()
    assert ok, 'no source yielded enough videos — see diagnostics above'

print('FINAL:', len(list(Path("dataset/crash").glob("*.mp4"))), 'crash |',
      len(list(Path("dataset/normal").glob("*.mp4"))), 'normal |',
      'event times:', len(json.load(open('dataset/event_times.json'))
                          if Path('dataset/event_times.json').exists() else []))

In [ ]:
# =============================================================================
# Cell 5 — WINDOW DATASET
# Positives: a window CENTERED ON THE COLLISION (dataset/event_times.json from
#            the acquisition cell). Clips without an event time fall back to a
#            late-in-clip window — a heuristic, so expect noisier labels there.
# Negatives: random windows from normal clips.
# =============================================================================
import cv2, json, random, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ---- preprocessing contract (mirrored by modal_app.py — do not drift) ----
N_FRAMES, SIDE = 16, 112
MEAN = np.array([0.43216, 0.394666, 0.37645],  dtype=np.float32)
STD  = np.array([0.22803, 0.22145, 0.216989], dtype=np.float32)
SPAN = 48                    # source frames per window (~1.6 s @ 30fps)

EVENT_TIMES = (json.load(open('dataset/event_times.json'))
               if Path('dataset/event_times.json').exists() else {})

def probe(path):
    """(n_frames, fps) — robust: many mp4s report FRAME_COUNT=0, so count by
    grabbing (this was the earlier num_samples=0 bug)."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0:
        n = 0
        while cap.grab():
            n += 1
    cap.release()
    return n, (fps if fps > 1 else 30.0)

def read_window(path, start, span):
    """N_FRAMES evenly sampled from [start, start+span) -> (C,T,H,W) tensor."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    for i in np.linspace(start, start + span - 1, N_FRAMES).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(cv2.resize(f, (SIDE, SIDE)), cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        raise ValueError(f'unreadable clip: {path}')
    while len(frames) < N_FRAMES:
        frames.append(frames[-1])
    arr = (np.stack(frames).astype(np.float32) / 255.0 - MEAN) / STD    # T,H,W,C
    return torch.from_numpy(arr).permute(3, 0, 1, 2)                     # C,T,H,W

class ClipWindows(Dataset):
    def __init__(self, crash, normal, neg_per_clip=2):
        self.items, self.skipped, self.no_event = [], 0, 0
        for p in crash:
            n, fps = probe(p)
            if n < N_FRAMES:
                self.skipped += 1; continue
            span = min(SPAN, n)
            if p.name in EVENT_TIMES:                    # center on the impact
                ev = int(EVENT_TIMES[p.name] * fps)
                start = ev - int(span * 0.5)
            else:                                        # heuristic fallback
                self.no_event += 1
                start = int(n * 0.75) - span
            self.items.append((p, max(0, min(start, n - span)), span, 1.0))
        for p in normal:
            n, _ = probe(p)
            if n < N_FRAMES:
                self.skipped += 1; continue
            span = min(SPAN, n)
            for _ in range(neg_per_clip):
                self.items.append((p, random.randint(0, max(0, n - span)), span, 0.0))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, start, span, y = self.items[i]
        return read_window(p, start, span), torch.tensor(y)

DATA = Path('dataset')
crash_clips  = sorted((DATA/'crash').glob('*.mp4'))
normal_clips = sorted((DATA/'normal').glob('*.mp4'))
print(f'{len(crash_clips)} crash mp4s, {len(normal_clips)} normal mp4s, '
      f'{len(EVENT_TIMES)} event times')
for p in (crash_clips[:2] + normal_clips[:2]):           # diagnostic
    n, fps = probe(p)
    print(f'  {p.name}: {n} frames @ {fps:.0f}fps, {p.stat().st_size // 1024} KB')

random.seed(7)
random.shuffle(crash_clips); random.shuffle(normal_clips)
cut_c, cut_n = int(len(crash_clips) * 0.85), int(len(normal_clips) * 0.85)
train_ds = ClipWindows(crash_clips[:cut_c], normal_clips[:cut_n])
val_ds   = ClipWindows(crash_clips[cut_c:], normal_clips[cut_n:])
print(f'train windows: {len(train_ds)} (skipped {train_ds.skipped}, '
      f'no-event-time positives {train_ds.no_event})   val windows: {len(val_ds)}')
assert len(train_ds) > 0 and len(val_ds) > 0, (
    '0 windows — dataset/ is empty or clips unreadable; re-run the acquisition '
    'cell and check its diagnostics (Colab wipes files on runtime reset).')
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights

model = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)
model.fc = torch.nn.Linear(model.fc.in_features, 1)   # single crash logit
model = model.to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M params')

In [ ]:
from sklearn.metrics import average_precision_score

EPOCHS = 2      # 2 epochs fits a free-tier session; raise only on Pro/Kaggle
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
lossf = torch.nn.BCEWithLogitsLoss()

def evaluate():
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in val_dl:
            p = torch.sigmoid(model(x.to(DEVICE)).squeeze(-1)).cpu()
            ys += y.tolist(); ps += p.tolist()
    ap  = average_precision_score(ys, ps)
    acc = float(np.mean([(p > 0.5) == (y > 0.5) for p, y in zip(ps, ys)]))
    return ap, acc

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        loss = lossf(model(x.to(DEVICE)).squeeze(-1), y.to(DEVICE))
        loss.backward(); opt.step(); tot += loss.item() * len(y)
    ap, acc = evaluate()
    print(f'epoch {ep}: train_loss={tot/len(train_ds):.4f}  val_AP={ap:.3f}  val_acc={acc:.3f}')

In [ ]:
import json
model.eval().cpu()
example = torch.randn(1, 3, N_FRAMES, SIDE, SIDE)
torch.jit.trace(model, example).save('crash_r3d18.ts')
json.dump({'n_frames': N_FRAMES, 'side': SIDE, 'mean': MEAN.tolist(),
           'std': STD.tolist(), 'arch': 'r3d_18', 'output': 'sigmoid(logit)'},
          open('preprocess_contract.json', 'w'), indent=1)
print('saved crash_r3d18.ts +', 'preprocess_contract.json')

# sanity: TorchScript output matches eager
ts = torch.jit.load('crash_r3d18.ts')
with torch.no_grad():
    assert torch.allclose(ts(example), model(example), atol=1e-4)
print('TorchScript sanity check OK')

try:
    from google.colab import files
    files.download('crash_r3d18.ts'); files.download('preprocess_contract.json')
except ImportError:
    pass  # not on Colab

## Next

1. `modal volume put crashguard-weights crash_r3d18.ts /crash_r3d18.ts`
2. `modal deploy modal_app.py` → copy the printed URL
3. `echo 'CRASH_MODEL_URL=<url>' >> .env` in the CrashGuard repo
4. `modal run modal_app.py::smoke --video data/clips/pileup.mp4` → expect high
   `crash_prob`; try `highway.mp4` → expect low
5. `make demo` — the server auto-enables the gated REASON stage when
   `CRASH_MODEL_URL` is set (`pipeline/reason.py`).